In [1]:
from pathlib import Path
import json, re

SCRIPTS = Path('..') / 'ejemplos_codigo'
print('[OK] Entorno B07 listo')
print(f'Scripts en: {SCRIPTS.resolve()}')

[OK] Entorno B07 listo
Scripts en: D:\documentos_gdrive\proyectos_claude_mcp\projects\formación_AI_springter\ejemplos_codigo


# Prompting como Programación Declarativa

Un LLM no se programa con código, se programa con lenguaje.  
El prompt es la nueva interfaz de programación, y disenarla bien es ingenieria, no redaccion.

**Marco mental de este bloque:**
```
TAREA  -->  PROMPT (instruccion + contexto + formato)  -->  OUTPUT controlado
```

**Actividad de calentamiento - El telefono descompuesto:**  
Participante A escribe un prompt para generar un email comercial específico.  
Participante B intenta mejorar el prompt sin ver el resultado.  
Se ejecutan ambos y se comparan.  
Conclusión: la precisión del prompt determina la calidad del resultado.

> **Antes de seguir:** ¿en qué se parece dar instrucciones a un LLM a dar instrucciones a una persona nueva en el equipo, y en qué se diferencia?

<details>
<summary>Orientación para el instructor (desplegar tras la reflexión)</summary>

**Una respuesta madura menciona al menos uno de estos elementos:**
- El LLM no puede pedir aclaraciones de forma autónoma ni inferir contexto implícito con la misma fiabilidad que una persona
- La persona nueva acumula contexto a lo largo del tiempo; el LLM empieza de cero en cada sesión (salvo memoria explícita)
- El LLM sigue las instrucciones de forma muy literal: la ambigüedad que un humano resolvería por sentido común produce resultados inesperados

**Si nadie responde, preguntar:**
"Si le dices a un compañero nuevo 'resume este email', ¿qué asume? ¿Asume lo mismo un LLM? ¿Qué tendrías que añadir al prompt para que el LLM hiciera lo que esperas?"

**Señal de comprensión:**
El alumno identifica que el prompting es más formal que una conversación natural: requiere hacer explícito lo que con una persona quedaría implícito. Si puede dar un ejemplo concreto de algo que un humano asumiría y un LLM no, ha captado la diferencia operativa.

</details>

## 7.1 Anatomía de un Prompt Efectivo

Un prompt tiene tres componentes fundamentales:

| Componente | Que define |
|---|---|
| **Instrucción** | Que tarea ejecutar, con que alcance, en que formato |
| **Contexto** | Información específica que el modelo necesita para resolver la tarea |
| **Formato de respuesta** | Como debe estructurarse la salida (JSON, lista, parrafo) |

**La diferencia entre un prompt mediocre y uno excelente esta en la precisión de la instrucción.**

```
Mal:  "Dime algo sobre los inventarios"
Bien: "Explica las tres métricas mas importantes para medir la salud de
       un inventario en retail, con una frase de definicion y un ejemplo
       numerico para cada una"
```

In [2]:
# Comparador de calidad de prompts
# Este ejercicio evalua estructuralmente cuanto especifica un prompt

def analizar_prompt(prompt: str) -> dict:
    """Evalua la completitud estructural de un prompt."""
    score = 0
    detalles = []

    # Instruccion especifica
    verbos_accion = ['explica', 'clasifica', 'extrae', 'genera', 'compara',
                     'resume', 'analiza', 'lista', 'calcula', 'describe']
    if any(v in prompt.lower() for v in verbos_accion):
        score += 25
        detalles.append('[OK] Contiene verbo de accion especifico')
    else:
        detalles.append('[X]  Sin verbo de accion claro')

    # Contexto
    if len(prompt) > 100:
        score += 25
        detalles.append('[OK] Longitud sugiere contexto presente')
    else:
        detalles.append('[!]  Prompt corto - posible falta de contexto')

    # Formato de salida
    formatos = ['json', 'lista', 'tabla', 'parrafo', 'bullet', 'formato',
                'responde en', 'devuelve', 'estructura']
    if any(f in prompt.lower() for f in formatos):
        score += 25
        detalles.append('[OK] Especifica formato de salida')
    else:
        detalles.append('[X]  Sin especificacion de formato')

    # Restricciones / alcance
    restricciones = ['no incluyas', 'solo', 'exclusivamente', 'maximo',
                     'minimo', 'sin', 'evita', 'limita']
    if any(r in prompt.lower() for r in restricciones):
        score += 25
        detalles.append('[OK] Define alcance o restricciones')
    else:
        detalles.append('[!]  Sin restricciones de alcance')

    nivel = 'POBRE' if score < 25 else 'BASICO' if score < 50 else 'BUENO' if score < 75 else 'EXCELENTE'
    return {'score': score, 'nivel': nivel, 'detalles': detalles}


# --- Prueba con dos prompts ---
prompt_malo = "Dime algo sobre los inventarios"

prompt_bueno = (
    "Explica las tres metricas mas importantes para medir la salud de un "
    "inventario en retail. Para cada metrica incluye: nombre, formula de calculo, "
    "y un ejemplo numerico con valores concretos. Responde en formato de lista "
    "numerada. No incluyas teoria general, solo las metricas."
)

for label, p in [('Prompt mediocre', prompt_malo), ('Prompt preciso', prompt_bueno)]:
    resultado = analizar_prompt(p)
    print(f'\n=== {label} ===')
    print(f'  Score: {resultado["score"]}/100  [{resultado["nivel"]}]')
    for d in resultado['detalles']:
        print(f'  {d}')


=== Prompt mediocre ===
  Score: 0/100  [POBRE]
  [X]  Sin verbo de accion claro
  [!]  Prompt corto -  posible falta de contexto
  [X]  Sin especificacion de formato
  [!]  Sin restricciones de alcance

=== Prompt preciso ===
  Score: 100/100  [EXCELENTE]
  [OK] Contiene verbo de accion especifico
  [OK] Longitud sugiere contexto presente
  [OK] Especifica formato de salida
  [OK] Define alcance o restricciones


## 7.2 Estrategias de Prompting

```
Zero-Shot  -->  One/Few-Shot  -->  Chain of Thought
     (sencillo)        (necesito ejemplos)      (requiere razonamiento)
```

**Zero-Shot**: instrucción directa sin ejemplos. Punto de partida.  
**Few-Shot**: 3-5 ejemplos resueltos que el modelo usa como patrón implícito.  
**Chain of Thought (CoT)**: "Razona paso a paso" - mejora dramaticamente tareas de cálculo y análisis multi-paso.

**Regla práctica**: si con 5 ejemplos no alcanzas la precisión necesaria,  
el problema probablemente requiere fine-tuning, RAG, o un pipeline mas sofisticado.

In [3]:
# Demostracion de Chain of Thought: calculo del punto de pedido
# El modelo con CoT explicita sus pasos - cada paso es verificable

import math

def calcular_punto_pedido(demanda_media_diaria: float,
                          lead_time_dias: int,
                          desviacion_estandar_diaria: float,
                          nivel_servicio_pct: float = 0.95) -> dict:
    """
    Replica el razonamiento CoT para calcular el punto de pedido.
    Cada paso esta explicito - como lo haria un modelo con CoT.
    """
    # Factores Z para niveles de servicio comunes
    z_table = {0.90: 1.282, 0.95: 1.645, 0.98: 2.054, 0.99: 2.326}
    z = z_table.get(nivel_servicio_pct, 1.645)

    pasos = {}

    # Paso 1: consumo durante el lead time
    consumo_lt = demanda_media_diaria * lead_time_dias
    pasos['1_consumo_lead_time'] = f'{demanda_media_diaria} * {lead_time_dias} = {consumo_lt:.2f} unidades'

    # Paso 2: desviacion durante el lead time
    desv_lt = desviacion_estandar_diaria * math.sqrt(lead_time_dias)
    pasos['2_desviacion_lead_time'] = f'{desviacion_estandar_diaria} * sqrt({lead_time_dias}) = {desv_lt:.2f} unidades'

    # Paso 3: factor Z
    pasos['3_factor_z'] = f'Nivel servicio {nivel_servicio_pct*100:.0f}% -> Z = {z}'

    # Paso 4: stock de seguridad
    stock_seguridad = z * desv_lt
    pasos['4_stock_seguridad'] = f'{z} * {desv_lt:.2f} = {stock_seguridad:.2f} unidades'

    # Paso 5: punto de pedido
    pp = math.ceil(consumo_lt + stock_seguridad)
    pasos['5_punto_pedido'] = f'{consumo_lt:.2f} + {stock_seguridad:.2f} = {pp} unidades (redondeado)'

    return {'pasos': pasos, 'resultado': pp}


# Problema del ejemplo de la seccion 7.2
print('Problema: almacen con 1200 unidades')
print('  Consumo semanal medio: 180 uds | Desv. std: 40 uds | Lead time: 2 semanas | NS: 95%')
print()

# La seccion convierte semanas a dias para el calculo
resultado = calcular_punto_pedido(
    demanda_media_diaria=180 / 7,   # convertir semanal a diario
    lead_time_dias=14,               # 2 semanas
    desviacion_estandar_diaria=40 / math.sqrt(7),
    nivel_servicio_pct=0.95
)

print('=== CoT - Razonamiento paso a paso ===')
for paso, detalle in resultado['pasos'].items():
    print(f'  Paso {paso}: {detalle}')
print(f'\n  PUNTO DE PEDIDO: {resultado["resultado"]} unidades')

Problema: almacen con 1200 unidades
  Consumo semanal medio: 180 uds | Desv. std: 40 uds | Lead time: 2 semanas | NS: 95%

=== CoT -  Razonamiento paso a paso ===
  Paso 1_consumo_lead_time: 25.714285714285715 * 14 = 360.00 unidades
  Paso 2_desviacion_lead_time: 15.118578920369089 * sqrt(14) = 56.57 unidades
  Paso 3_factor_z: Nivel servicio 95% -> Z = 1.645
  Paso 4_stock_seguridad: 1.645 * 56.57 = 93.06 unidades
  Paso 5_punto_pedido: 360.00 + 93.06 = 454 unidades (redondeado)

  PUNTO DE PEDIDO: 454 unidades


## 7.3 Plantillas Reutilizables

Un prompt de producción no se escribe desde cero cada vez.  
Las plantillas con variables permiten estandarizar y escalar.

**En la empresa**: las plantillas se versionan como código, se almacenan en Git,  
y se despliegan como parte de la configuración de los servicios que consumen la API del LLM.  
Un cambio en el prompt es un cambio de comportamiento - se gestiona con el mismo rigor que un cambio de código.

In [4]:
# Sistema de plantillas de prompts con validacion de variables

class PromptTemplate:
    """Plantilla de prompt con variables tipadas y validacion."""

    def __init__(self, name: str, version: str, template: str,
                 variables_schema: dict, metadata: dict = None):
        self.name = name
        self.version = version
        self.template = template
        self.variables_schema = variables_schema
        self.metadata = metadata or {}

    def render(self, variables: dict) -> str:
        # Validar variables requeridas
        for var, schema in self.variables_schema.items():
            if schema.get('required') and var not in variables:
                raise ValueError(f"Variable requerida '{var}' no proporcionada")
        return self.template.format(**variables)

    def __repr__(self):
        return f'PromptTemplate({self.name} v{self.version})'


# Plantilla de clasificacion de tickets (v3) para la empresa
clasificador_tickets_v3 = PromptTemplate(
    name='clasificador_tickets',
    version='3.0.1',
    template=(
        'Eres un analista de soporte de nivel 2 de la empresa.\n'
        'Clasifica el siguiente ticket.\n\n'
        'Modulo: {modulo}\n'
        'Titulo: {titulo}\n'
        'Descripcion: {descripcion}\n\n'
        'Responde en JSON con la siguiente estructura:\n'
        '{{"categoria": "bug|feature|pregunta|config",\n'
        '  "prioridad": "critica|alta|media|baja",\n'
        '  "confianza": 0.0-1.0}}'
    ),
    variables_schema={
        'modulo':      {'type': 'string', 'required': True, 'example': 'Inventarios'},
        'titulo':      {'type': 'string', 'required': True},
        'descripcion': {'type': 'string', 'required': True},
    },
    metadata={
        'autor': 'equipo-ia',
        'modelo_target': 'claude-haiku-4-5-20251001',
        'precision_validada': 0.92,
    }
)

# Renderizar con un ticket de ejemplo
ticket_ejemplo = {
    'modulo':      'Inventarios',
    'titulo':      'Error al exportar informe a Excel',
    'descripcion': 'Al pulsar el boton de exportar, la pagina se queda en loading y a los 30 segundos aparece un error 500.',
}

prompt_renderizado = clasificador_tickets_v3.render(ticket_ejemplo)
print(f'=== {clasificador_tickets_v3} ===')
print()
print(prompt_renderizado)
print()
print(f'Precision validada: {clasificador_tickets_v3.metadata["precision_validada"]*100:.0f}%')
print(f'Modelo target: {clasificador_tickets_v3.metadata["modelo_target"]}')

=== PromptTemplate(clasificador_tickets v3.0.1) ===

Eres un analista de soporte de nivel 2 de la empresa.
Clasifica el siguiente ticket.

Modulo: Inventarios
Titulo: Error al exportar informe a Excel
Descripcion: Al pulsar el boton de exportar, la pagina se queda en loading y a los 30 segundos aparece un error 500.

Responde en JSON con la siguiente estructura:
{"categoria": "bug|feature|pregunta|config",
  "prioridad": "critica|alta|media|baja",
  "confianza": 0.0-1.0}

Precision validada: 92%
Modelo target: claude-haiku-4-5-20251001


## 7.4 Salidas Estructuradas y Validación

Un LLM que devuelve texto libre requiere parsing manual.  
Un LLM que devuelve JSON estructurado se integra directamente en el flujo de datos.

**En producción**: las respuestas del LLM deben validarse contra un esquema  
antes de procesarse. El modelo puede generar:
- JSON malformado
- Campos omitidos
- Valores fuera del rango esperado

**Estrategia de reintentos**: enviar la respuesta malformada de vuelta al modelo  
con instrucción "corrige el siguiente JSON para que cumpla el esquema"  
suele resolver el problema en un segundo intento.

In [5]:
# Validacion de salidas JSON de un LLM
# Simula el flujo de produccion con esquema y manejo de errores

import json

# Esquema esperado para la respuesta del clasificador
ESQUEMA_CLASIFICADOR = {
    'required': ['categoria', 'prioridad', 'confianza'],
    'categoria': {'values': ['bug', 'feature', 'pregunta', 'config']},
    'prioridad':  {'values': ['critica', 'alta', 'media', 'baja']},
    'confianza':  {'type': float, 'min': 0.0, 'max': 1.0},
}

def validar_respuesta_llm(respuesta_texto: str, esquema: dict) -> tuple:
    """
    Valida una respuesta JSON del LLM contra un esquema.
    Retorna (datos, lista_errores).
    """
    errores = []

    # Intentar parsear JSON
    try:
        datos = json.loads(respuesta_texto)
    except json.JSONDecodeError as e:
        return None, [f'JSON invalido: {e}']

    # Verificar campos requeridos
    for campo in esquema.get('required', []):
        if campo not in datos:
            errores.append(f'Campo requerido ausente: {campo}')

    # Verificar valores permitidos
    for campo, reglas in esquema.items():
        if campo == 'required' or campo not in datos:
            continue
        if 'values' in reglas and datos[campo] not in reglas['values']:
            errores.append(f'Valor invalido en {campo}: "{datos[campo]}" (permitidos: {reglas["values"]})')
        if 'type' in reglas:
            try:
                val = reglas['type'](datos[campo])
                if 'min' in reglas and val < reglas['min']:
                    errores.append(f'{campo} < {reglas["min"]}')
                if 'max' in reglas and val > reglas['max']:
                    errores.append(f'{campo} > {reglas["max"]}')
            except (TypeError, ValueError):
                errores.append(f'{campo} no es de tipo {reglas["type"].__name__}')

    return datos, errores


# Casos de prueba: respuestas simuladas del LLM
casos_respuesta = [
    ('{"categoria": "bug", "prioridad": "alta", "confianza": 0.91}',
     'Respuesta correcta'),
    ('{"categoria": "bug", "prioridad": "urgente", "confianza": 0.91}',
     'Valor fuera de enum (prioridad: urgente)'),
    ('{"categoria": "bug", "confianza": 0.91}',
     'Campo faltante (prioridad)'),
    ('El ticket parece un bug de alta prioridad.',
     'JSON malformado'),
    ('{"categoria": "bug", "prioridad": "alta", "confianza": 1.5}',
     'Confianza fuera de rango (>1.0)'),
]

print('=== Validacion de respuestas del LLM ===')
for respuesta, descripcion in casos_respuesta:
    datos, errores = validar_respuesta_llm(respuesta, ESQUEMA_CLASIFICADOR)
    estado = '[OK]' if not errores else '[X] '
    print(f'\n{estado} {descripcion}')
    if errores:
        for e in errores:
            print(f'       Error: {e}')
    else:
        print(f'       Resultado: {datos}')

=== Validacion de respuestas del LLM ===

[OK] Respuesta correcta
       Resultado: {'categoria': 'bug', 'prioridad': 'alta', 'confianza': 0.91}

[X]  Valor fuera de enum (prioridad: urgente)
       Error: Valor invalido en prioridad: "urgente" (permitidos: ['critica', 'alta', 'media', 'baja'])

[X]  Campo faltante (prioridad)
       Error: Campo requerido ausente: prioridad

[X]  JSON malformado
       Error: JSON invalido: Expecting value: line 1 column 1 (char 0)

[X]  Confianza fuera de rango (>1.0)
       Error: confianza > 1.0


## 7.6 Prompts como Productos

Un prompt que se usa una vez es un experimento.  
Un prompt que se usa en producción es un producto.

**Ciclo de vida de un prompt en producción:**
```
1. DESARROLLO   -> escribir y iterar con ejemplos
2. EVALUACIÓN   -> dataset de test, métricas, comparar con version anterior
3. STAGING      -> trafico real, sin impacto en usuarios
4. PRODUCCIÓN   -> rollout gradual (10% -> 50% -> 100%)
5. MONITORIZACIÓN -> detectar degradacion, alertas, nuevos ejemplos de test
```

**Para la empresa**: el equipo ya gestiona releases con CI/CD.  
Aplicar el mismo rigor a los prompts es una extension natural de sus prácticas.

In [6]:
# Framework de testing de prompts
# Evalua la precision de una plantilla contra un dataset de test

# Dataset de evaluacion para el clasificador de tickets
DATASET_TEST = [
    {
        'input': {'modulo': 'Inventarios', 'titulo': 'Error al exportar informe',
                  'descripcion': 'Al pulsar exportar a Excel sale error 500'},
        'expected': {'categoria': 'bug', 'prioridad': 'alta'},
    },
    {
        'input': {'modulo': 'Ventas', 'titulo': 'Descuentos por volumen',
                  'descripcion': 'Nos gustaria poder configurar descuentos automaticos por volumen'},
        'expected': {'categoria': 'feature', 'prioridad': 'media'},
    },
    {
        'input': {'modulo': 'Compras', 'titulo': 'Como cambiar el proveedor de un pedido',
                  'descripcion': 'Un usuario pregunta si puede cambiar el proveedor de un pedido ya confirmado'},
        'expected': {'categoria': 'pregunta', 'prioridad': 'baja'},
    },
    {
        'input': {'modulo': 'Usuarios', 'titulo': 'Permisos mal configurados en rol Admin',
                  'descripcion': 'El rol Admin no tiene acceso al modulo de Contabilidad segun la configuracion actual'},
        'expected': {'categoria': 'config', 'prioridad': 'alta'},
    },
]

def simular_clasificacion(input_data: dict) -> dict:
    """
    Simulacion determinista de un clasificador de LLM.
    En produccion esto seria una llamada a la API del modelo.
    """
    desc = input_data['descripcion'].lower()
    if any(w in desc for w in ['error', 'fallo', 'bug', 'sale', 'no funciona']):
        return {'categoria': 'bug', 'prioridad': 'alta', 'confianza': 0.88}
    if any(w in desc for w in ['gustaria', 'podria', 'queremos', 'funcionalidad']):
        return {'categoria': 'feature', 'prioridad': 'media', 'confianza': 0.85}
    if any(w in desc for w in ['como', 'pregunta', 'si puede']):
        return {'categoria': 'pregunta', 'prioridad': 'baja', 'confianza': 0.82}
    if any(w in desc for w in ['configuracion', 'rol', 'permiso']):
        return {'categoria': 'config', 'prioridad': 'alta', 'confianza': 0.79}
    return {'categoria': 'pregunta', 'prioridad': 'baja', 'confianza': 0.5}


def evaluar_prompt(dataset: list, umbral: float = 0.75) -> float:
    correctos = 0
    errores = []
    for caso in dataset:
        resultado = simular_clasificacion(caso['input'])
        if resultado['categoria'] == caso['expected']['categoria']:
            correctos += 1
        else:
            errores.append({
                'titulo': caso['input']['titulo'],
                'expected': caso['expected']['categoria'],
                'got': resultado['categoria'],
            })
    precision = correctos / len(dataset)
    estado = '[OK]' if precision >= umbral else '[X] '
    print(f'{estado} Precision: {precision*100:.0f}% ({correctos}/{len(dataset)}) | Umbral: {umbral*100:.0f}%')
    if errores:
        print('  Errores:')
        for e in errores:
            print(f'    Ticket: "{e["titulo"]}" | Expected: {e["expected"]} | Got: {e["got"]}')
    return precision


print('=== Evaluacion del clasificador de tickets v3.0.1 ===')
precision = evaluar_prompt(DATASET_TEST, umbral=0.75)
print()
print('Nota: en produccion, el dataset debe tener 50-100 ejemplos reales.')
print('Un dataset pequeno da una estimacion ruidosa de la precision real.')

=== Evaluacion del clasificador de tickets v3.0.1 ===
[OK] Precision: 100% (4/4) | Umbral: 75%

Nota: en produccion, el dataset debe tener 50-100 ejemplos reales.
Un dataset pequeno da una estimacion ruidosa de la precision real.


---
## 7. Ejercicio de Decisión: ¿usarias IA aquí?

### Caso: el prompt de generación de propuestas comerciales

El equipo de ventas de la empresa usa un prompt para generar borradores de propuestas
comerciales a partir de la información de la RFP del cliente.
Lleva 3 meses en producción. Un comercial senior informa que "el 20% de las propuestas
salen mal y hay que rehacerlas completamente".

Un desarrollador propone añadir 500 palabras de instrucciones extra al prompt
para cubrir los casos que fallan.

---

**Pregunta 1 - Diagnóstico antes de actuar**
¿Como medirías si ese 20% es un problema del prompt, de la calidad de las RFPs
de entrada, o del criterio del comercial que las evalua?

**Pregunta 2 - La propuesta de añadir instrucciones**
¿Es buena idea añadir 500 palabras al prompt? ¿Que podria salir mal?
¿Que alternativa propones?

**Pregunta 3 - Mejorar sin romper lo que funciona**
¿Como cambiarias el prompt para mejorar el 20% sin degradar el 80% que ya funciona?
Describe el proceso, no solo el resultado.

**Pregunta 4 - Cuando cambiar de enfoque**
¿Cuando dejarias de intentar mejorar el prompt y optarias por un modelo fine-tuneado?
¿Que condición concreta te haria tomar esa decisión?

---
*Escribe tus respuestas en la celda siguiente.*

### Mis respuestas

**Pregunta 1 - Diagnóstico antes de actuar:**

*(escribe aquí)*

**Pregunta 2 - La propuesta de añadir instrucciones:**

*(escribe aquí)*

**Pregunta 3 - Mejorar sin romper lo que funciona:**

*(escribe aquí)*

**Pregunta 4 - Cuando cambiar de enfoque:**

*(escribe aquí)*

---

<!--
CRITERIOS DE Evaluación (para el instructor)

Pregunta 1 - Diagnóstico antes de actuar:
Respuesta correcta: necesitas un criterio objetivo de "propuesta mala" antes de medir.
Preguntas de diagnóstico validas:
 - Que caracteristicas tienen en comun las RFPs de los casos que fallan?
 - Es el mismo comercial quien evalua las buenas y las malas, o hay diferentes criterios?
 - En que parte de la propuesta falla: estructura, tono, datos técnicos, precio?
Insuficiente: "miraria las propuestas malas" sin un método sistemático.
Incorrecto: proponer soluciones antes de entender donde esta el problema.

Pregunta 2 - La propuesta de añadir instrucciones:
Problema principal: un prompt mas largo no siempre es mejor. Puede:
 - Crear contradicciones internas entre instrucciones
 - Sobrecargar el contexto útil del modelo
 - Hacer el prompt fragil (un cambio en un caso rompe otros)
Alternativa valida: few-shot examples de los casos que fallan, o separar el prompt en pasos
(primero estructura, luego tono, luego precio), o usar un modelo mas capaz.
Insuficiente: solo decir "no es buena idea" sin proponer alternativa.

Pregunta 3 - Mejorar sin romper lo que funciona:
Proceso correcto:
  1. Aislar los casos que fallan (muestra representativa)
  2. Experimentar con el prompt en un entorno controlado solo con esos casos
  3. Verificar que el nuevo prompt no degrada los casos que ya funcionan
  4. Desplegar en A/B o con revisión humana durante un período
Insuficiente: proponer cambiar el prompt y "ver que pasa".

Pregunta 4 - Cuando cambiar de enfoque:
Condición concreta valida: cuando la calidad máxima alcanzable con prompting sigue siendo
insuficiente despues de haber probado sistematicamente distintas variantes,
o cuando el volumen justifica el coste del fine-tuning.
También valido: cuando el problema requiere un estilo muy específico que el modelo base no sigue
por mucho que se le instruya.
Incorrecto: proponer fine-tuning como primera opción sin agotar el prompting.
-->


## Resumen del Bloque 7

```
Prompt efectivo = Instrucción precisa + Contexto específico + Formato definido
```

| Estrategia | Cuando usar |
|---|---|
| Zero-Shot | Tarea simple, bien definida |
| Few-Shot (3-5 ej) | Cuando el patrón importa o hay casos de borde |
| Chain of Thought | Razonamiento, cálculo, análisis multi-paso |
| Plantilla con variables | Producción: estandarizar y versionar |
| Salida JSON + validación | Integración en pipelines de software |

**Decisión clave**: un prompt de producción tiene dataset de evaluación,  
métricas documentadas y un proceso de despliegue controlado.  
No esta terminado cuando funciona en 5 ejemplos.

**Siguiente bloque**: Limites, Riesgos y Gobernanza - 
que pasa cuando el sistema falla y como controlarlo.